# NB_SLV_POI_AMENITY

RECOMPUTE `lh_vv_silver.dbo.slv_poi_amenity`. Khớp `source_flags_json` với từ điển google theo `business_sector` của `slv_poi_entity_non_enrich`.


In [ ]:
p_exec_id = ""
p_run_id = ""
p_pl_name = ""
p_trg_fq = ""
p_pinned_json = ""
p_mode = "RUN"
p_force = "false"
p_history_limit = "1000"


In [ ]:
%run NB_LIB_TRANSFORM_SLV_GLD


In [ ]:
# BEGIN_AMENITY_MATCH
import re

FLAG_KEYS = ("servesBreakfast", "servesBrunch", "servesLunch", "servesDinner", "servesBeer", "servesWine",
             "servesCocktails", "servesDessert", "servesCoffee", "servesVegetarianFood", "allowsDogs",
             "goodForChildren", "goodForGroups", "goodForWatchingSports", "liveMusic", "menuForChildren",
             "outdoorSeating", "restroom", "reservable", "takeout", "delivery", "dineIn", "curbsidePickup")
OBJECT_KEYS = ("parkingOptions", "paymentOptions", "accessibilityOptions", "evChargeOptions", "fuelOptions")
COND_RE = re.compile(r"([A-Za-z][A-Za-z0-9]*)(?:\.([A-Za-z][A-Za-z0-9]*))? = (true|false)")


def resolve_condition(raw):
    """Điều kiện -> (path trong source_flags_json, giá trị) hoặc None. Ví dụ outdoorSeating = true -> (flags.outdoorSeating, true)."""
    m = COND_RE.fullmatch((raw or "").strip())
    if m is None:
        return None
    head, leaf, value = m.groups()
    if leaf is None:
        return (f"flags.{head}", value) if head in FLAG_KEYS else None
    if head == "flags":
        return (f"flags.{leaf}", value) if leaf in FLAG_KEYS else None
    return (f"{head}.{leaf}", value) if head in OBJECT_KEYS else None
# END_AMENITY_MATCH


In [ ]:
from collections import Counter

GOOGLE = "google"
COND_SCHEMA = ("business_sector string, amenity_group string, amenity_raw string, amenity string, "
               "amenity_vi string, _path string, _v string")
OUT_COLS = ("poi_id", "business_sector", "amenity_group", "amenity_raw", "amenity", "amenity_vi")


def build(ctx: NodeContext) -> DataFrame:
    # Từ điển google active -> điều kiện khớp được. Điều kiện không đúng dạng path = true/false thì bỏ qua.
    rows = (ctx.input("dict")
            .where((F.lower(F.trim("source_name")) == GOOGLE) & (F.col("is_active") == F.lit(True)))
            .select("business_sector", "amenity_group", "amenity_raw", "amenity", "amenity_vi").collect())
    conds = [tuple(r) + hit for r in rows if (hit := resolve_condition(r.amenity_raw))]
    dup = sorted(k for k, n in Counter(c[:3] for c in conds).items() if n > 1)
    if dup:
        raise QualityGateError(f"ref_dictionary_amenities: điều kiện google trùng khoá {dup}")
    if not conds:
        raise QualityGateError("ref_dictionary_amenities: không có điều kiện google khớp được")
    cond = F.broadcast(spark.createDataFrame(conds, COND_SCHEMA))
    # source_flags_json = {"flags": {...}, "parkingOptions": {...} | null, ...}; object null -> không nổ dòng.
    src = ctx.input("amenity").select("poi_id", "source_flags_json")
    top = src.select("poi_id", F.explode(F.from_json("source_flags_json", "map<string,string>")).alias("_top", "_obj"))
    flat = (top.select("poi_id", "_top", F.explode(F.from_json("_obj", "map<string,string>")).alias("_leaf", "_v"))
               .select("poi_id", F.concat_ws(".", "_top", "_leaf").alias("_path"), "_v"))
    ent = ctx.input("entity").select("poi_id", "business_sector")
    out = flat.join(ent, "poi_id").join(cond, ["business_sector", "_path", "_v"]).select(*OUT_COLS)
    src_rows = src.count()
    unparsed = src.where(F.col("source_flags_json").isNotNull()
                         & F.from_json("source_flags_json", "map<string,string>").isNull()).count()
    joined = src.join(ent, "poi_id").count()
    n = out.count()
    ctx.metrics.update(src_rows=src_rows, pois_joined=joined, pois_without_entity=src_rows - joined,
                       unparsed_json=unparsed, dict_google_active=len(rows),
                       conditions_matchable=len(conds), conditions_skipped=len(rows) - len(conds), rows=n)
    # Không có dòng khớp là trạng thái hợp lệ: DataFrame rỗng để RECOMPUTE soft-delete amenity cũ.
    return out


SPEC = NodeSpec(
    trg_tbl="slv_poi_amenity",
    inputs={"amenity": "slv_3p_poi_amenity", "entity": "slv_poi_entity_non_enrich", "dict": "ref_dictionary_amenities"},
    pk=("poi_id", "business_sector", "amenity_group", "amenity_raw"),
    build=build,
    required_inputs=("amenity", "entity", "dict"),
    write_mode="MERGE",
)


In [ ]:
RES = run_node(SPEC, p_exec_id, p_run_id, p_pinned_json, p_mode, p_force, p_pl_name, p_trg_fq, p_history_limit)


In [ ]:
notebookutils.notebook.exit(finish_node(RES))
